# 🚦 Fine-tuning YOLOv8 — Trafic de Douala
## Dataset : lynkeus / vehicle-detection-mgjdd (Roboflow)

**Projet Tutoré 4ème année — IA/Informatique**  
Auteurs : NGUEBOU TEMGOUA Rayan, BEKOU Adrien  
Encadrants : M. NDJE MAN D.F., M. KAZE Roger (3iL / Labo MIA)

---
### Ce notebook réalise :
1. ✅ Installation des dépendances
2. ✅ Téléchargement du dataset depuis Roboflow
3. ✅ Fine-tuning de YOLOv8n sur 6 classes de véhicules
4. ✅ Évaluation et visualisation des résultats
5. ✅ Export du modèle pour notre système de surveillance

> ⚠️ **Avant de commencer** : Allez dans `Exécution → Modifier le type d'exécution → GPU (T4)

## 📋 ÉTAPE 0 — Vérification du GPU

In [ ]:

# Vérification que le GPU T4 est bien disponible
import torch

print('='*55)
print('  VÉRIFICATION DE L\'ENVIRONNEMENT')
print('='*55)
print(f'  PyTorch version  : {torch.__version__}')
print(f'  GPU disponible   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU détecté      : {gpu_name}')
    print(f'  Mémoire GPU      : {gpu_mem:.1f} Go')
    print('  ✅ Prêt pour le fine-tuning !')
else:
    print('  ❌ Aucun GPU détecté !')
    print('  → Allez dans Exécution > Modifier le type d\'exécution > GPU')
print('='*55)

## 📦 ÉTAPE 1 — Installation des dépendances

In [ ]:
# Installation de YOLOv8 (Ultralytics) et du SDK Roboflow
!pip install ultralytics roboflow --quiet

# Vérification
import ultralytics
ultralytics.checks()
print('\n✅ Ultralytics YOLOv8 installé avec succès !')

## 🔑 ÉTAPE 2 — Connexion à Roboflow & Téléchargement du Dataset

> **Obtenez votre clé API gratuite ici :**  
> https://app.roboflow.com → Settings → API Keys

In [ ]:
from roboflow import Roboflow

# ============================================================
# ⚠️  REMPLACEZ PAR VOTRE VRAIE CLÉ API ROBOFLOW
# ============================================================
ROBOFLOW_API_KEY = "dTz2FtQq2l7cseTysWT5"
# ============================================================

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Accès au dataset lynkeus/vehicle-detection-mgjdd
project = rf.workspace("lynkeus").project("vehicle-detection-mgjdd")

# Téléchargement de la version la plus récente au format YOLOv8
# Listez les versions disponibles :
versions = project.versions()
print("Versions disponibles :")
for v in versions:
    print(f"  Version {v.version} — {v.splits}")

# Téléchargement (remplacez le numéro de version si nécessaire)
dataset = project.version(3).download("yolov8")

print(f"\n✅ Dataset téléchargé dans : {dataset.location}")

## 🔍 ÉTAPE 3 — Exploration du Dataset

In [ ]:
import os
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

# Lecture du fichier data.yaml
yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print('='*55)
print('  INFORMATIONS DU DATASET')
print('='*55)
print(f"  Nombre de classes : {data_config['nc']}")
print(f"  Classes           : {data_config['names']}")

# Comptage des images par split
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset.location, split, 'images')
    if os.path.exists(split_path):
        n = len(os.listdir(split_path))
        print(f"  Images {split:<8}  : {n}")
print('='*55)

# Correspondance avec le contexte Douala
print("\n  Correspondance avec le trafic de Douala :")
douala_map = {
    'car':        'Voitures classiques',
    'motorbike':  'Benskins (motos-taxis)',
    'bus':        'Cars clandos (minibus)',
    'truck':      'Camions',
    'microbus':   'Taxis collectifs',
    'pickup-van': 'Pick-up / Fourgonnettes',
}
for cls in data_config['names']:
    doula_name = douala_map.get(cls, cls)
    print(f"    {cls:<15} → {doula_name}")

In [ ]:
# Visualisation de quelques images annotées du dataset
import random

COLORS = {
    0: '#00C800',  # car        — vert
    1: '#FFA500',  # motorbike  — orange
    2: '#3232FF',  # bus        — bleu
    3: '#DC0000',  # truck      — rouge
    4: '#AA00FF',  # microbus   — violet
    5: '#00AAFF',  # pickup-van — cyan
}

train_img_dir = os.path.join(dataset.location, 'train', 'images')
train_lbl_dir = os.path.join(dataset.location, 'train', 'labels')
img_files = [f for f in os.listdir(train_img_dir) if f.endswith(('.jpg','.jpeg','.png'))]

sample = random.sample(img_files, min(6, len(img_files)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Exemples du Dataset — lynkeus/vehicle-detection\n(Annotations YOLOv8)',
             fontsize=14, fontweight='bold')

for ax, img_file in zip(axes.flatten(), sample):
    img_path = os.path.join(train_img_dir, img_file)
    lbl_path = os.path.join(train_lbl_dir, img_file.rsplit('.', 1)[0] + '.txt')

    img = Image.open(img_path)
    W, H = img.size
    ax.imshow(img)

    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    # Conversion YOLO → pixels
                    x1 = (cx - bw/2) * W
                    y1 = (cy - bh/2) * H
                    pw = bw * W
                    ph = bh * H
                    color = COLORS.get(cls_id, '#FFFFFF')
                    rect = patches.Rectangle((x1,y1), pw, ph,
                                             linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    cls_name = data_config['names'][cls_id] if cls_id < len(data_config['names']) else str(cls_id)
                    ax.text(x1, y1-4, cls_name, color='white', fontsize=8,
                            bbox=dict(facecolor=color, alpha=0.7, pad=1))
    ax.axis('off')
    ax.set_title(img_file[:30], fontsize=9)

plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisation sauvegardée : dataset_samples.png')

In [ ]:
import os

# Voir exactement ce que Roboflow a téléchargé
print("=== CONTENU DU DOSSIER DATASET ===")
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:  # Affiche 5 fichiers max par dossier
            print(f"{indent}  📄 {f}")

print(f"\nChemin du dataset : {dataset.location}")
print(f"yaml_path actuel  : {yaml_path}")

In [ ]:
import os
import yaml

# ── Étape 1 : Trouver le vrai yaml_path ──────────────────────
yaml_path = None
for root, dirs, files in os.walk(dataset.location):
    for f in files:
        if f == 'data.yaml':
            yaml_path = os.path.join(root, f)
            break

print(f"✅ data.yaml trouvé : {yaml_path}")

# ── Étape 2 : Lire et afficher le contenu du yaml ────────────
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"\nContenu du data.yaml :")
print(f"  nc    : {data_config.get('nc')}")
print(f"  names : {data_config.get('names')}")
print(f"  train : {data_config.get('train')}")
print(f"  val   : {data_config.get('val')}")
print(f"  test  : {data_config.get('test', 'absent')}")

# ── Étape 3 : Corriger les chemins dans le yaml ───────────────
# Roboflow génère parfois des chemins relatifs qui ne correspondent
# pas à la structure réelle — on les corrige avec des chemins absolus
base = dataset.location
data_config_fixed = data_config.copy()

train_path_found = None
# First, find the actual train path
for candidate in ['train/images', 'train']:
    full_path = os.path.join(base, candidate)
    if os.path.exists(full_path) and os.path.isdir(full_path) and len(os.listdir(full_path)) > 0:
        train_path_found = full_path
        print(f"  ✅ train trouvé → {full_path}")
        data_config_fixed['train'] = full_path
        break
if not train_path_found:
    print(f"  ⚠️  train non trouvé")

# Then, try to find val and test, or default to train if only train is available
for split_key, split_candidates in [
    ('val',   ['valid/images', 'val/images', 'validation/images', 'valid', 'val']),
    ('test',  ['test/images', 'test']),
]:
    path_found = False
    for candidate in split_candidates:
        full_path = os.path.join(base, candidate)
        if os.path.exists(full_path) and os.path.isdir(full_path) and len(os.listdir(full_path)) > 0:
            data_config_fixed[split_key] = full_path
            print(f"  ✅ {split_key} trouvé → {full_path}")
            path_found = True
            break
    if not path_found:
        if train_path_found:
            # Fallback to train path if val/test is not found but train is
            data_config_fixed[split_key] = train_path_found
            print(f"  ⚠️  {split_key} non trouvé, utilisant le chemin de train → {train_path_found}")
        else:
            # If even train is not found, just leave it as it was (or set to None/empty, but Ultralytics might crash)
            print(f"  ⚠️  {split_key} non trouvé et aucun chemin de train valide pour fallback.")

# Sauvegarder le yaml corrigé
yaml_fixed_path = os.path.join(base, 'data_fixed.yaml')
with open(yaml_fixed_path, 'w') as f:
    yaml.dump(data_config_fixed, f, default_flow_style=False)

yaml_path = yaml_fixed_path  # ← On utilise ce yaml corrigé

print(f"\n✅ yaml corrigé sauvegardé : {yaml_path}")
print(f"\nContenu corrigé :")
with open(yaml_path, 'r') as f:
    print(f.read())

## 🔧 ÉTAPE 4 — Fine-tuning YOLOv8n

### Paramètres choisis et justification

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| `model` | `yolov8n.pt` | Nano — rapide, <25 Mo, fonctionne sans GPU |
| `epochs` | `50` | Suffisant pour fine-tuning, évite le surapprentissage |
| `imgsz` | `640` | Standard YOLOv8, bon compromis vitesse/précision |
| `batch` | `16` | Adapté à la mémoire GPU T4 (16 Go) |
| `patience` | `10` | Arrêt anticipé si pas d'amélioration |
| `augment` | `True` | Rotation, flip, luminosité → robustesse au contexte Douala |

In [ ]:
from ultralytics import YOLO

# Chargement du modèle pré-entraîné sur COCO
model = YOLO('yolov8n.pt')

print('='*55)
print('  LANCEMENT DU FINE-TUNING')
print('='*55)
print(f'  Modèle de base : yolov8n.pt (pré-entraîné COCO)')
print(f'  Dataset        : lynkeus/vehicle-detection-mgjdd')
print(f'  Epochs         : 50')
print(f'  Image size     : 640x640')
print(f'  Batch size     : 16')
print('  Démarrage...')
print('='*55)

# Fine-tuning
results = model.train(
    data=yaml_path,           # Chemin vers data.yaml du dataset
    epochs=50,                 # Nombre d'époques
    imgsz=640,                 # Taille des images
    batch=16,                  # Taille du batch (réduire à 8 si mémoire insuffisante)
    patience=10,               # Early stopping : arrêt si pas d'amélioration après 10 époques
    device=0,                  # GPU 0 (T4 sur Colab)
    project='runs/train',      # Dossier de sauvegarde
    name='yolov8_douala',      # Nom de l'expérience
    exist_ok=True,

    # ── Augmentations pour robustesse au contexte Douala ──
    # Variations d'éclairage (soleil tropical, contre-jour)
    hsv_h=0.015,               # Variation teinte
    hsv_s=0.7,                 # Variation saturation
    hsv_v=0.4,                 # Variation luminosité
    # Géométriques (angles de caméra variés)
    degrees=5.0,               # Rotation légère
    translate=0.1,             # Translation
    scale=0.5,                 # Zoom
    fliplr=0.5,                # Flip horizontal
    # Occlusions (trafic dense de Douala)
    mosaic=1.0,                # Mosaïque 4 images (simule l'occlusion)
    mixup=0.1,                 # Mixup pour robustesse
    copy_paste=0.1,            # Copy-paste augmentation

    # ── Hyperparamètres ──
    lr0=0.01,                  # Taux d'apprentissage initial
    lrf=0.01,                  # Taux d'apprentissage final (lr0 * lrf)
    momentum=0.937,
    weight_decay=0.0005,

    # ── Logging ──
    plots=True,                # Générer les courbes d'entraînement
    save=True,                 # Sauvegarder les checkpoints
    save_period=3,      # Sauvegarde tous les 3 epochs
    verbose=True,
)

print('\n✅ Fine-tuning terminé !')
print(f'   Meilleur modèle : runs/train/yolov8_douala/weights/best.pt')

In [ ]:
import shutil

shutil.make_archive(
    "yolo_results",
    'zip',
    "/kaggle/working/runs"
)

print("ZIP créé avec succès")

In [ ]:
import shutil
from IPython.display import FileLink

# Création du fichier ZIP
shutil.make_archive(
    "/kaggle/working/yolo_results",
    'zip',
    "/kaggle/working/runs"
)

print("ZIP créé avec succès")

# Afficher lien de téléchargement
display(FileLink("/kaggle/working/yolo_results.zip"))

In [ ]:
import os

print(os.path.exists("/kaggle/working/yolo_results.zip"))

In [ ]:
import os

for f in os.listdir("/kaggle/working"):
    print(f)

## 📊 ÉTAPE 5 — Évaluation des Performances

In [ ]:
# Chargement du meilleur modèle
best_model = YOLO('/kaggle/working/runs/detect/runs/train/yolov8_douala/weights/best.pt')

# Évaluation sur le jeu de validation
print('Évaluation sur le jeu de validation...')
metrics = best_model.val(
    data=yaml_path,
    split='val',
    verbose=True
)

print('\n' + '='*60)
print('  RÉSULTATS D\'ÉVALUATION — Modèle Fine-tuné')
print('='*60)
print(f'  mAP@50      : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@50-95   : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Précision   : {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)')
print(f'  Rappel      : {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)')
print('='*60)
print()
print('  Performances par classe :')
class_names = data_config['names']
for i, cls in enumerate(class_names):
    try:
        ap = metrics.box.ap50[i]
        print(f'    {cls:<15} AP@50 : {ap:.4f} ({ap*100:.1f}%)')
    except:
        pass
print('='*60)

In [ ]:
# Visualisation des courbes d'entraînement
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

curves_path = '/kaggle/working/runs/detect/runs/train/yolov8_douala/results.png'
if os.path.exists(curves_path):
    img = mpimg.imread(curves_path)
    plt.figure(figsize=(20, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Courbes d\'entraînement YOLOv8n — Dataset Lynkeus', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Courbes non disponibles — vérifiez le dossier runs/train/yolov8_douala/')

## 🧪 ÉTAPE 6 — Test sur Images / Vidéo

In [ ]:
import os

base = "/kaggle/working/vehicle-detection-3"

for root, dirs, files in os.walk(base):
    print(root)

In [ ]:
# Test sur quelques images du jeu de test
import random
from PIL import Image
import numpy as np

test_img_dir = os.path.join(dataset.location, '/kaggle/working/vehicle-detection-3/train/', 'images')
if not os.path.exists(test_img_dir):
    test_img_dir = os.path.join(dataset.location, 'valid', 'images')

test_imgs = [os.path.join(test_img_dir, f)
             for f in os.listdir(test_img_dir)
             if f.endswith(('.jpg','.jpeg','.png'))]

# Inférence sur 6 images
sample_imgs = random.sample(test_imgs, min(6, len(test_imgs)))

# Couleurs par classe
CLASS_COLORS_BGR = [
    (0, 200, 0),    # car        — vert
    (0, 165, 255),  # motorbike  — orange
    (255, 50, 50),  # bus        — bleu
    (0, 0, 220),    # truck      — rouge
    (255, 0, 170),  # microbus   — violet
    (255, 200, 0),  # pickup-van — cyan
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Inférence — Modèle Fine-tuné YOLOv8n (lynkeus dataset)',
             fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), sample_imgs):
    result = best_model.predict(
        img_path,
        conf=0.35,
        verbose=False
    )[0]

    # Image annotée par YOLOv8
    annotated = result.plot()
    annotated_rgb = annotated[:, :, ::-1]  # BGR → RGB
    ax.imshow(annotated_rgb)
    ax.axis('off')
    n_det = len(result.boxes)
    ax.set_title(f'{n_det} véhicule(s) détecté(s)', fontsize=10)

plt.tight_layout()
plt.savefig('inference_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Résultats sauvegardés : inference_results.png')

In [ ]:
from IPython.display import Image, display

display(Image(filename='/kaggle/working/inference_results.png'))

## 💾 ÉTAPE 7 — Comparaison Avant/Après Fine-tuning

In [ ]:
# Comparaison modèle COCO de base vs modèle fine-tuné
base_model   = YOLO('yolov8n.pt')             # Modèle de base (COCO)
finetuned    = YOLO('/kaggle/working/runs/detect/runs/train/yolov8_douala/weights/best.pt')  # Fine-tuné

test_image = sample_imgs[0]

# Inférence des deux modèles
res_base = base_model.predict(test_image, conf=0.35,
    classes=[2,3,5,7],  # Classes COCO véhicules
    verbose=False)[0]
res_ft   = finetuned.predict(test_image, conf=0.35, verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Comparaison : Modèle COCO de base vs Modèle Fine-tuné',
             fontsize=14, fontweight='bold')

axes[0].imshow(res_base.plot()[:,:,::-1])
axes[0].set_title(f'YOLOv8n COCO (base)\n{len(res_base.boxes)} détections',
                   fontsize=12, color='red')
axes[0].axis('off')

axes[1].imshow(res_ft.plot()[:,:,::-1])
axes[1].set_title(f'YOLOv8n Fine-tuné (lynkeus)\n{len(res_ft.boxes)} détections',
                   fontsize=12, color='green')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('comparaison_avant_apres.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparaison sauvegardée : comparaison_avant_apres.png')

## 📥 ÉTAPE 8 — Export & Téléchargement du Modèle

In [ ]:
import shutil

# Chemin du meilleur modèle
best_weights = '/kaggle/working/runs/detect/runs/train/yolov8_douala/weights/best.pt'
last_weights = '/kaggle/working/runs/detect/runs/train/yolov8_douala/weights/last.pt'

# Copie dans un dossier propre pour l'export
os.makedirs('export', exist_ok=True)
shutil.copy(best_weights, 'export/yolov8_douala_best.pt')

# Taille du fichier
size_mb = os.path.getsize('export/yolov8_douala_best.pt') / 1e6

print('='*55)
print('  EXPORT DU MODÈLE FINE-TUNÉ')
print('='*55)
print(f'  Fichier  : export/yolov8_douala_best.pt')
print(f'  Taille   : {size_mb:.1f} Mo')
print(f'  Classes  : {data_config["names"]}')
print('='*55)
print()
print('  Instructions d\'utilisation dans notre système :')
print('  ─────────────────────────────────────────────')
print('  1. Téléchargez export/yolov8_douala_best.pt')
print('  2. Placez-le dans le dossier models/ du projet')
print('  3. Dans config.py, changez :')
print('     MODEL_PATH = "models/yolov8_douala_best.pt"')
print('  4. Mettez à jour VEHICLE_CLASSES dans config.py')
print('     (voir cellule suivante)')

# Téléchargement depuis Colab
try:
    from google.colab import files
    print('\n  Téléchargement automatique en cours...')
    files.download('export/yolov8_douala_best.pt')
    print('  ✅ Modèle téléchargé !')
except:
    print('\n  (Téléchargement manuel : clic droit sur le fichier dans Files)')

## ⚙️ ÉTAPE 9 — Nouveau config.py pour le modèle fine-tuné

In [ ]:
# Génération du nouveau config.py adapté au modèle fine-tuné

class_names = data_config['names']

config_content = f'''# ============================================================
# config.py — Configuration pour le modèle FINE-TUNÉ
# Dataset : lynkeus/vehicle-detection-mgjdd
# ============================================================

# Source vidéo
VIDEO_SOURCE = 0  # 0=webcam, ou chemin vers votre vidéo

# Modèle fine-tuné (remplace yolov8n.pt)
MODEL_PATH = "/kaggle/working/Exportation/yolov8_douala_best.pt"

# Seuil de confiance
CONFIDENCE_THRESHOLD = 0.35
IOU_THRESHOLD = 0.45

# Classes du modèle fine-tuné (lynkeus dataset)
# IDs dans l\'ordre du data.yaml
VEHICLE_CLASSES = {{
    {', '.join([f'{i}: "{n.capitalize()}"' for i, n in enumerate(class_names)])}
}}

# Toutes les classes sont des véhicules
DETECT_ALL_CLASSES = True  # Si True, détecte toutes les classes du modèle

# Couleurs (BGR)
CLASS_COLORS = {{
    0: (0, 200, 0),      # car        — vert
    1: (0, 165, 255),    # motorbike  — orange
    2: (255, 50, 50),    # bus        — bleu
    3: (0, 0, 220),      # truck      — rouge
    4: (255, 0, 200),    # microbus   — violet
    5: (255, 200, 0),    # pickup-van — cyan
}}

# DeepSORT
MAX_AGE = 30
N_INIT  = 3
MAX_COSINE_DISTANCE = 0.4
NN_BUDGET = 100

# Affichage
SHOW_FPS          = True
SHOW_TRAJECTORIES = True
TRAJECTORY_LENGTH = 40
SHOW_COUNTER      = True
DISPLAY_WIDTH     = 1280
DISPLAY_HEIGHT    = 720

# Enregistrement
SAVE_OUTPUT_VIDEO = True
OUTPUT_VIDEO_PATH = "outputs/detection_finetuned.mp4"

# Ligne de comptage
COUNTING_LINE_Y     = 0.55
COUNTING_LINE_COLOR = (0, 255, 255)
'''

with open('export/config_finetuned.py', 'w') as f:
    f.write(config_content)

print('✅ Nouveau config.py généré : export/config_finetuned.py')
print()
print('Contenu :')
print('─' * 50)
print(config_content)

## ✅ Récapitulatif Final

In [ ]:
print('='*60)
print('  ✅ FINE-TUNING TERMINÉ AVEC SUCCÈS')
print('='*60)
print()
print('  Fichiers à télécharger depuis Colab :')
print('  ├── export/yolov8_douala_best.pt    (modèle fine-tuné)')
print('  ├── export/config_finetuned.py      (nouvelle config)')
print('  ├── dataset_samples.png             (images du dataset)')
print('  ├── inference_results.png           (résultats d\'inférence)')
print('  └── comparaison_avant_apres.png     (avant/après)')
print()
print('  Pour utiliser dans votre système :')
print('  1. Copiez yolov8_douala_best.pt → models/')
print('  2. Remplacez config.py par config_finetuned.py')
print('  3. Lancez :')
print('     python module1_detection_suivi.py --source data/video.mp4')
print()
print('  Classes détectées par le modèle :')
for i, cls in enumerate(class_names):
    print(f'    [{i}] {cls}')
print('='*60)